# Analysis of temporal analysis
This code was written by Sai and Danyka Byrnes

In [168]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from config import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

#### Answer the following questions based on the timeseries data:
1) Percentage of data missing between the start and end date.
  
2) The length between start and end datetime (hour) 

3) longest continous recored between the start and end date.  

In [169]:
# List of possible parameters
parameters = [
    "WTemp_C", "SpC_uScm", "DO_mgL", "pH",
    "Turb_FNU", "Turb_NTU", "NO3_mgNL",
    "fDOM_QSU", "fDOM_RFU", "DOC_mgL",
    "PO4_mgL", "Chla_ugL", "Chla_RFU",
    "PC_ugL", "PC_RFU"
]

PARAM_COLORS = {
    'DO_mgL':    {'fill': '#FEDCBB', 'edge': '#7F2704'},
    'SpC_uScm':  {'fill': '#E2EDF8', 'edge': '#08306B'},
    'Turb_FNU':  {'fill': '#CEECC8', 'edge': '#00431A'},
}

param_pretty = {"WTemp_C":"Temp (°C)",
                "SpC_uScm":"SpC (µS/cm)",
                "DO_mgL":"DO (mg/L)",
                "pH":"pH",
                "Turb_FNU":"Turbidity (FNU)",
                "Turb_NTU":"Turbidity (NTU)",
                "NO3_mgNL":"NO₃-N (mg/L)",
                "fDOM_QSU":"fDOM (QSU)",
                "fDOM_RFU":"fDOM (RFU)",
                "DOC_mgL":"DOC (mg/L)",
                "PO4_mgL":"PO₄ (mg/L)",
                "Chla_ugL":"Chl-a (µg/L)",
                "Chla_RFU":"Chl-a (RFU)",
                "PC_ugL":"PC (µg/L)",
                "PC_RFU":"PC (RFU)",
               }

param_subset = ['SpC_uScm','DO_mgL','Turb_FNU']

In [170]:
# Store all results
results = []

# Loop through each CSV
for file in os.listdir(water_quality_filepath):
    if not file.endswith(".csv"):
        continue

    file_path = os.path.join(water_quality_filepath, file)
    
    # Extract STREAM_ID from filename (remove .csv)
    stream_id = os.path.splitext(file)[0]

    df = pd.read_csv(file_path, low_memory=False) # mixed data types in df, maybe worth double checking

    if "DateTime" not in df.columns:
        continue

    # Convert DateTime
    df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
    df = df.sort_values("DateTime")

    station_result = {"STREAM_ID": stream_id}

    for param in parameters:
        if param in df.columns:

            df_solute = df[['DateTime', param]].dropna(subset=param)
            
            # Removing erroneous values 
            df_solute.loc[df_solute[param] < 0, param] = np.nan
            
            # Dropping NA then finding the max and min date.
            start = df_solute["DateTime"].min()
            end = df_solute["DateTime"].max()
            
            full_range = pd.date_range(start=start, end=end, freq="h")
            
            # Reindex to full hourly timeline to gapfill hours in the middle that may be empty.
            df_solute = df_solute.set_index("DateTime").reindex(full_range)
            
            # Dropping NA then finding the max and min date.
            series = df_solute[param]
            
            # 1 Length between start and end
            length_hours = len(full_range)
            
            # 2 Percentage Missing
            missing_count = series.isna().sum() # this only counts NAs in the middle of timeseries
            percent_missing = (missing_count / length_hours) * 100

            # 3 Longest continuous record (non-missing streak)
            not_na = series.notna().astype(int)
            groups = (not_na.diff() != 0).cumsum()
            streak_lengths = not_na.groupby(groups).sum()
            longest_streak = streak_lengths.max() if len(streak_lengths) > 0 else 0

            # 4  total hours available
            actual_hours = series.notna().sum()

            # 5 Seasonal gaps. 
            monthly_pct = series.groupby(series.index.to_period("M")).apply(
                lambda x: x.isna().mean() * 100)
            median_by_month = monthly_pct.groupby(monthly_pct.index.month).median()

            # The choice of replacing nan with 100 is not arbitrary. The reason I picked 100 instead of NaN is because
            # seasonal I am plotting missingness against MAT to understand whether cold-climate stations have systematic winter gaps
            median_by_month = median_by_month.reindex(range(1, 13), fill_value=100)

            # populat the list with the new summary metrics
            station_result[f"{param}_pct_missing"] = round(percent_missing, 2)
            station_result[f"{param}_length_hr"] = length_hours
            station_result[f"{param}_longest_hr"] = int(longest_streak)
            station_result[f"{param}_actual_hr"] = int(actual_hours)

            for month_num in range(1, 13):
                col = f"{param}_pct_missing_{month_num:02d}"
                if month_num in median_by_month.index:
                    station_result[col] = round(median_by_month[month_num], 2)
                else:
                    station_result[col] = np.nan

        else:
            station_result[f"{param}_pct_missing"] = np.nan
            station_result[f"{param}_length_hr"] = np.nan
            station_result[f"{param}_longest_hr"] = np.nan
            station_result[f"{param}_actual_hr"] = np.nan
            for month_num in range(1, 13):
                station_result[f"{param}_pct_missing_{month_num:02d}"] = np.nan
            
    results.append(station_result)

# Convert to dataframe
temporal_summary = pd.DataFrame(results)
temporal_summary.head()

,STREAM_ID,WTemp_C_pct_missing,WTemp_C_length_hr,WTemp_C_longest_hr,WTemp_C_actual_hr,WTemp_C_pct_missing_01,WTemp_C_pct_missing_02,WTemp_C_pct_missing_03,WTemp_C_pct_missing_04,WTemp_C_pct_missing_05,WTemp_C_pct_missing_06,WTemp_C_pct_missing_07,WTemp_C_pct_missing_08,WTemp_C_pct_missing_09,WTemp_C_pct_missing_10,WTemp_C_pct_missing_11,WTemp_C_pct_missing_12,SpC_uScm_pct_missing,SpC_uScm_length_hr,SpC_uScm_longest_hr,SpC_uScm_actual_hr,SpC_uScm_pct_missing_01,SpC_uScm_pct_missing_02,SpC_uScm_pct_missing_03,SpC_uScm_pct_missing_04,SpC_uScm_pct_missing_05,SpC_uScm_pct_missing_06,SpC_uScm_pct_missing_07,SpC_uScm_pct_missing_08,SpC_uScm_pct_missing_09,SpC_uScm_pct_missing_10,SpC_uScm_pct_missing_11,SpC_uScm_pct_missing_12,DO_mgL_pct_missing,DO_mgL_length_hr,DO_mgL_longest_hr,DO_mgL_actual_hr,DO_mgL_pct_missing_01,DO_mgL_pct_missing_02,DO_mgL_pct_missing_03,DO_mgL_pct_missing_04,DO_mgL_pct_missing_05,DO_mgL_pct_missing_06,DO_mgL_pct_missing_07,DO_mgL_pct_missing_08,DO_mgL_pct_missing_09,DO_mgL_pct_missing_10,DO_mgL_pct_missing_11,DO_mgL_pct_missing_12,pH_pct_missing,pH_length_hr,pH_longest_hr,pH_actual_hr,pH_pct_missing_01,pH_pct_missing_02,pH_pct_missing_03,pH_pct_missing_04,pH_pct_missing_05,pH_pct_missing_06,pH_pct_missing_07,pH_pct_missing_08,pH_pct_missing_09,pH_pct_missing_10,pH_pct_missing_11,pH_pct_missing_12,Turb_FNU_pct_missing,Turb_FNU_length_hr,Turb_FNU_longest_hr,Turb_FNU_actual_hr,Turb_FNU_pct_missing_01,Turb_FNU_pct_missing_02,Turb_FNU_pct_missing_03,Turb_FNU_pct_missing_04,Turb_FNU_pct_missing_05,Turb_FNU_pct_missing_06,Turb_FNU_pct_missing_07,Turb_FNU_pct_missing_08,Turb_FNU_pct_missing_09,Turb_FNU_pct_missing_10,Turb_FNU_pct_missing_11,Turb_FNU_pct_missing_12,Turb_NTU_pct_missing,Turb_NTU_length_hr,Turb_NTU_longest_hr,Turb_NTU_actual_hr,Turb_NTU_pct_missing_01,Turb_NTU_pct_missing_02,Turb_NTU_pct_missing_03,Turb_NTU_pct_missing_04,Turb_NTU_pct_missing_05,Turb_NTU_pct_missing_06,Turb_NTU_pct_missing_07,Turb_NTU_pct_missing_08,Turb_NTU_pct_missing_09,Turb_NTU_pct_missing_10,Turb_NTU_pct_missing_11,Turb_NTU_pct_missing_12,NO3_mgNL_pct_missing,NO3_mgNL_length_hr,NO3_mgNL_longest_hr,NO3_mgNL_actual_hr,NO3_mgNL_pct_missing_01,NO3_mgNL_pct_missing_02,NO3_mgNL_pct_missing_03,NO3_mgNL_pct_missing_04,NO3_mgNL_pct_missing_05,NO3_mgNL_pct_missing_06,NO3_mgNL_pct_missing_07,NO3_mgNL_pct_missing_08,NO3_mgNL_pct_missing_09,NO3_mgNL_pct_missing_10,NO3_mgNL_pct_missing_11,NO3_mgNL_pct_missing_12,fDOM_QSU_pct_missing,fDOM_QSU_length_hr,fDOM_QSU_longest_hr,fDOM_QSU_actual_hr,fDOM_QSU_pct_missing_01,fDOM_QSU_pct_missing_02,fDOM_QSU_pct_missing_03,fDOM_QSU_pct_missing_04,fDOM_QSU_pct_missing_05,fDOM_QSU_pct_missing_06,fDOM_QSU_pct_missing_07,fDOM_QSU_pct_missing_08,fDOM_QSU_pct_missing_09,fDOM_QSU_pct_missing_10,fDOM_QSU_pct_missing_11,fDOM_QSU_pct_missing_12,fDOM_RFU_pct_missing,fDOM_RFU_length_hr,fDOM_RFU_longest_hr,fDOM_RFU_actual_hr,fDOM_RFU_pct_missing_01,fDOM_RFU_pct_missing_02,fDOM_RFU_pct_missing_03,fDOM_RFU_pct_missing_04,fDOM_RFU_pct_missing_05,fDOM_RFU_pct_missing_06,fDOM_RFU_pct_missing_07,fDOM_RFU_pct_missing_08,fDOM_RFU_pct_missing_09,fDOM_RFU_pct_missing_10,fDOM_RFU_pct_missing_11,fDOM_RFU_pct_missing_12,DOC_mgL_pct_missing,DOC_mgL_length_hr,DOC_mgL_longest_hr,DOC_mgL_actual_hr,DOC_mgL_pct_missing_01,DOC_mgL_pct_missing_02,DOC_mgL_pct_missing_03,DOC_mgL_pct_missing_04,DOC_mgL_pct_missing_05,DOC_mgL_pct_missing_06,DOC_mgL_pct_missing_07,DOC_mgL_pct_missing_08,DOC_mgL_pct_missing_09,DOC_mgL_pct_missing_10,DOC_mgL_pct_missing_11,DOC_mgL_pct_missing_12,PO4_mgL_pct_missing,PO4_mgL_length_hr,PO4_mgL_longest_hr,PO4_mgL_actual_hr,PO4_mgL_pct_missing_01,PO4_mgL_pct_missing_02,PO4_mgL_pct_missing_03,PO4_mgL_pct_missing_04,PO4_mgL_pct_missing_05,PO4_mgL_pct_missing_06,PO4_mgL_pct_missing_07,PO4_mgL_pct_missing_08,PO4_mgL_pct_missing_09,PO4_mgL_pct_missing_10,PO4_mgL_pct_missing_11,PO4_mgL_pct_missing_12,Chla_ugL_pct_missing,Chla_ugL_length_hr,Chla_ugL_longest_hr,Chla_ugL_actual_hr,Chla_ugL_pct_missing_01,Chla_ugL_pct_missing_02,C

In [171]:
# Merging the temporal data derived by Sai
# Identify ID/metadata columns (keep these as-is)
id_cols = ['STREAM_ID']

# Reshape
rows = []
for temp_param in parameters:
    temp = temporal_summary[id_cols].copy()
    temp['parameter'] = temp_param
    temp['pct_missing'] = temporal_summary[f'{temp_param}_pct_missing']
    temp['length_hr'] = temporal_summary[f'{temp_param}_length_hr']
    temp['longest_hr'] = temporal_summary[f'{temp_param}_longest_hr']
    temp['actual_hr'] = temporal_summary[f'{temp_param}_actual_hr']

    # This parameter is the median percentage of the months across the record period that is missing data data. 
    # If the value is 100, it means that in at least one year, that month was in the timeseries, but it missing data
    # If the value is NaN, then the month never appeared in the timeseries.
    for month_num in range(1, 13):
        temp[f'pct_missing_{month_num:02d}'] = temporal_summary[f'{temp_param}_pct_missing_{month_num:02d}']
    rows.append(temp)

temporal_summary_long = (
    pd.concat(rows, ignore_index=True)
    .query('parameter in @parameters')
    .dropna(how='all', subset=['pct_missing', 'length_hr', 'longest_hr', 'actual_hr'])
)

temporal_summary_long.to_csv(OUTPUT_filepath+'temporal_statistics.csv', index=False)

PermissionError: [Errno 13] Permission denied: '../OUTPUT/temporal_statistics.csv'

In [ ]:
# For now, just set 100 and NaNs both to NaN because I feel like these are corner cases. 
month_cols = [f'pct_missing_{m:02d}' for m in range(1, 13)]
temporal_summary_long[month_cols] = temporal_summary_long[month_cols].replace(np.nan, 100)

temporal_summary_long.head()

In [ ]:
# Processing metadata, isolating specific parameters
md_wide = pd.read_csv(metadata_filepath+"metadata.csv", dtype = {'sourceID': str})
print(f"Full dataset size: {md_wide.shape[0]}")

for param in param_subset:
    md_wide[param] = md_wide['WQ_parameters'].str.contains(param)

# Removing the stations without data
md_wide = md_wide[(md_wide['SpC_uScm']) | (md_wide['DO_mgL']) | (md_wide['Turb_FNU'])]
print(f"Filtered dataset size: {md_wide.shape[0]}")

In [ ]:
md_wide.head()

## Seasonal Gaps

In [ ]:
# Reading and filtering 
basinatlas = pd.read_csv(basin_atlas_filepath+"static_basinatlas_catchment_characteristics.csv")
basinatlas = basinatlas[['STREAM_ID',
                         'tmp_dc_syr', # mean annual temperature
                        ]]

param_dict = {'tmp_dc_syr':'MAT'}

# Converting to true values
basinatlas['tmp_dc_syr'] = basinatlas['tmp_dc_syr']/10

# merging MAT with temporal variables
temperature_data = pd.merge(temporal_summary_long, basinatlas, on='STREAM_ID')
temperature_data.head()

In [ ]:
month_cols = [f'pct_missing_{m:02d}' for m in range(1, 13)]
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(9, 2.5), sharey=True)

for i, param in enumerate(param_subset):
    temp_data_p = temperature_data[temperature_data['parameter'] == param]
    n_stations = len(temp_data_p)
    
    freq_100 = [(temp_data_p[mcol] == 100).sum() / n_stations * 100 for mcol in month_cols]
    
    axes[i].bar(month_labels, freq_100)
    axes[i].set_title(param)
    axes[i].set_ylabel('% of stations with no data' if i == 0 else '')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylim(0, 50)

fig.tight_layout()

In [ ]:
# Plotting temporal variables against temperature. 
for param in param_subset:
    # First drop the variables we are not looking at.
    temp_data_p = temperature_data[temperature_data['parameter'].isin([param])]
    # Plot the solutes we are
    fig, axes = plt.subplots(1,3, figsize=(7, 2))
    axes[0].scatter(temp_data_p['tmp_dc_syr'], temp_data_p['length_hr']/24)
    axes[0].set_xlabel('MAT')
    axes[0].set_ylabel('Length of record (day)')
    
    axes[1].scatter(temp_data_p['tmp_dc_syr'], temp_data_p['longest_hr']/24)
    axes[1].set_xlabel('MAT')
    axes[1].set_ylabel('Longest period (day)')
    
    axes[2].scatter(temp_data_p['tmp_dc_syr'], temp_data_p['pct_missing'])
    axes[2].set_xlabel('MAT')
    axes[2].set_ylabel('Percent Missing')
    fig.tight_layout()

In [ ]:
# Plotting temporal variables against temperature. 
month_cols = [f'pct_missing_{m:02d}' for m in range(1, 13)]
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(3, 12, figsize=(24, 6), sharey=True, sharex=True)
for row, param in enumerate(param_subset):
    temp_data_p = temperature_data[temperature_data['parameter'] == param]
    
    for col, mcol in enumerate(month_cols):
        ax = axes[row, col]
        ax.scatter(temp_data_p['tmp_dc_syr'], temp_data_p[mcol], s=5, alpha=0.5)

        if row == 0:
            ax.set_title(month_labels[col], fontsize=8)
        if col == 0:
            ax.set_ylabel(f"(% missing) for {param}", fontsize=8)
        if row == 2:
            ax.set_xlabel('MAT', fontsize=7)

fig.tight_layout()

In [ ]:
from scipy.stats import gaussian_kde
from statsmodels.nonparametric.smoothers_lowess import lowess

In [ ]:
months = {'Jan': 'pct_missing_01', 'Apr': 'pct_missing_04', 'Jul': 'pct_missing_07', 'Oct': 'pct_missing_10'}
colors = {'Jan': '#a6cee3', 'Apr': '#1f78b4', 'Jul': '#b2df8a', 'Oct': '#33a02c'}

fig, axes = plt.subplots(2, 3, figsize=(12, 5), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

for i, param in enumerate(param_subset):
    temp_data_p = temperature_data[temperature_data['parameter'] == param]
    
    for label, mcol in months.items():
        
        x = temp_data_p['tmp_dc_syr'].values
        y = temp_data_p[mcol].values
        
        # LOWESS smooth
        smoothed = lowess(y, x, frac=0.6, missing='drop')
        axes[0, i].plot(smoothed[:, 0], smoothed[:, 1], label=label, color=colors[label], linewidth=2)
    
    # Histogram of MAT distribution
    x_all = temp_data_p['tmp_dc_syr'].dropna().values
    axes[1, i].hist(x_all, bins=20, 
                     facecolor=PARAM_COLORS[param]['fill'], 
                     edgecolor=PARAM_COLORS[param]['edge'], 
                     alpha=0.8)
    axes[1, i].set_xlabel('MAT')
    axes[1, i].set_ylabel('No. of stations' if i == 0 else '')
    
    # Aesthetic figure features
    if i == 0:
        axes[0, i].set_ylabel('Percent of record missing')
    axes[0, i].legend(fontsize=7)

    # set axis limits
    axes[1, i].set_xlim(5, 20)
    axes[0, i].set_xlim(5, 20)
    axes[1, i].set_ylim(0, 75)
    axes[0, i].set_ylim(0, 100)
    
    axes[0,i].text(5.25, 85, param_pretty[param])

fig.tight_layout()
plt.savefig(f'../OUTPUT/temporal_patterns/parameter_temporal_pattern_MAT.png', dpi=600, bbox_inches='tight')